# ☕🤖 RoboBarista – **die komplette Simulation in einem Colab**

Ein Notebook für unsere **Artly-inspirierte Kaffeeinsel** mit integriertem Mühlen-Tamper, **Kunststoffbechern**, sieben Andockstellen, drei AprilTag-Kameras und einem FAIRINO FR5.

Du musst nichts von Hand installieren oder mehrere Notebooks öffnen: **Laufzeit → Alle ausführen**. CPU genügt.

| Schritt | Was wir ausprobieren | Was du danach siehst |
|---|---|---|
| 1 | Vollständige Kaffeeinsel, AprilTag-Sichtbarkeit, geometrische Sicherheitsabstände, Tamper | Übersichtsvideo, Grundriss und Stationen |
| 2 | **Sieben frei angefahrene Vorpositionen** mit kontrollierten Arm- und Werkzeug-Kollisionen | FR5-Video, gesperrte/erlaubte Stationen |
| 3 | **15 FPS**: Zufällig erzeugte erlaubte Bewegung wird von drei RGB-D-Kameras **optisch aus AprilTags rekonstruiert** und dann per FR5-IK nachgefahren | Film **links Kamera / rechts Roboter**, XYZ-Vergleich, echte Kameramesswerte |
| **Neu: 3a + 3b** | **15 FPS** AprilTag-Messung → kontinuierliche 6D-Splines → lokal mit **Ruckig** jerk-begrenzte Gelenkbewegung | 100-Hz-Bewegungsdaten, Ruckig-FR5-Film, Kurven und Sicherheitsprotokoll |
| 4 | Die gemessenen FR5-Gelenkziele werden erstmals mit den **sechs echten MuJoCo-Drehmomentmotoren** und `mj_step` simuliert | Motorvideo, Trackingfehler, Grenz-/Kontaktbericht |

**Was nicht vorgetäuscht wird:** Die Bewegung ist **keine menschliche Demonstration**; auch der Kannen-Griff ist noch virtuell befestigt, keine Flüssigkeit fließt, und die Maschinen sind einfache Näherungsformen. Andockkontakte sind noch nicht freigegeben. Das *physikalische* Motorergebnis ist zunächst ein Versuchsmodell, keine Aussage über die reale FR5-Genauigkeit. Der Roboter muss bei Kollisionen oder ungültigen Kamera-Messungen abbrechen. Das vollständige optische Ergebnis bleibt getrennt von der Simulations-Wahrheit.

**Neu im selben Notebook:** Ein vollständiges **11-stufiges Espresso–Milch–Latte-Art-Storyboard** mit kinematisch geprüften FR5-Fahrten (Gerätekontakt und Flüssigkeit sind noch symbolisch), plus **unabhängige Raumrekonstruktion aus drei RGB-D-Bildern** mit Vergleich gegen die wirkliche MuJoCo-Geometrie. Diese neuen Experimente können separat ausgeführt werden und scheitern sicher, wenn ihre Prüfungen nicht bestehen.


In [ ]:
# 0. Einheitliche, reproduzierbare Umgebung
from pathlib import Path
import os, sys, subprocess, json, zipfile
from IPython.display import display, Video, Image, FileLink
import pandas as pd
repo=Path('/content/robobarista-unified')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1',
       'https://github.com/Coffee-Tobi/robocasa.git',str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
os.chdir(repo)
print('Git SHA:',subprocess.check_output(
    ['git','-C',str(repo),'rev-parse','--short','HEAD'],text=True).strip())
subprocess.run(['apt-get','-qq','update'],check=True)
subprocess.run(['apt-get','-qq','install','-y',
    'libosmesa6-dev','libgl1','libglx-mesa0','libgl1-mesa-dri','ffmpeg'],check=True)
subprocess.run([sys.executable,'-m','pip','-q','install',
    '-r',str(repo/'requirements.txt')],check=True)
env=os.environ.copy()
env['MUJOCO_GL']='osmesa'
env['PYOPENGL_PLATFORM']='osmesa'
runs=repo/'runs'/'complete_colab'
runs.mkdir(parents=True,exist_ok=True)
print('Alles installiert – CPU Rendering (OSMesa)')
# Ein konstanter Start-Seed, damit die getestete sichere Bewegung wiederholbar ist.
# Du kannst ihn später ändern; dann werden alle Kollisionen/Sichtbarkeiten
# erneut geprüft und die Demonstration ggf. sicher abgebrochen.
SEED = 14
# Realistische langsame, zeitgestreckte Motorprobe statt sofortigem qpos-Setzen.
TIME_SCALE = 4.0

def run_experiment(title,module,flags):
    print('\\n===',title,'===',flush=True)
    cmd=[sys.executable,'-u','-m',module]+flags
    p=subprocess.Popen(cmd,cwd=str(repo),env=env,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
        text=True,errors='replace',bufsize=1)
    for line in p.stdout:
        print(line,end='',flush=True)
    code=p.wait()
    if code:
        raise RuntimeError(f'{title} ist fehlgeschlagen (Exit {code}). '
            'Der Test hat deshalb nichts als Erfolg ausgegeben. '
            'Bitte die letzten Zeilen hier im Chat zeigen.')
    print('Bestanden:',title,flush=True)

def show_video(folder,base,fallback=None,width=900):
    f=folder/(base+'.mp4')
    if f.is_file():
        try:
            display(Video(str(f),embed=True,width=width))
            return
        except (TypeError,ValueError,OSError) as e:
            print('MP4-Wiedergabe nicht möglich:',e)
    if fallback:
        alt=folder/(fallback+'.mp4')
        if alt.is_file():
            try:
                display(Video(str(alt),embed=True,width=width))
                return
            except (TypeError,ValueError,OSError):
                pass
    for name in (base,fallback):
        if not name:continue
        img=folder/(name+'.gif')
        if img.exists():
            display(Image(filename=str(img),format='gif'))
            return
    print('Keine abspielbare Videodatei gefunden in',folder)


## 1. Unsere reale **virtuelle** Kaffeeinsel

FR5 zentral auf der Insel; Mühle **mit eingebautem Tamper**; Siebträger-Ablage; Espressomaschine (getrennte Brühgruppe, Tropfschale und Milchdüse); Milchspender; Milchkanne; Kunststoffbecher; Rinser. Die Kamera-Tag-Zuordnung und die sieben präzisen *nominellen* Andockpunkte werden als JSON gespeichert. Die Millimeterkoordinaten sind **Planungswerte, keine real vermessenen Gerätewerte**.


In [ ]:
workcell=runs/'workcell'
run_experiment('Kaffeeinsel + Kameras + Muehlen-Tamper',
    'robobarista.cell_demo',
    ['--frames','8','--fps','5','--width','920','--height','660',
     '--reach','--output',str(workcell)])
show_video(workcell,'workcell_demo')
if (workcell/'layout_topdown.png').exists():
    display(Image(filename=str(workcell/'layout_topdown.png')))
s=json.loads((workcell/'summary.json').read_text())
print('Muehlen-Tamper motorisch (isolierter Test):',
    s['tamper_motor_diagnostics']['down_success'],
    s['tamper_motor_diagnostics']['return_success'])
print('Sicherheitsfilter blockiert geraden Kannenflug durch Maschine:',
    s['no_contact_policy_test']['forbidden_pass_through_blocked'])
dock=json.loads((workcell/'workspace_constraints.json').read_text())['docks']
display(pd.DataFrame([
    {'Dock':name,'Werkzeug':v['tool'],
     'X mm':round(v['position_m'][0]*1000),
     'Y mm':round(v['position_m'][1]*1000),
     'Z mm':round(v['position_m'][2]*1000)}
    for name,v in dock.items()]))


## NEU: Raum aus den drei RGB-D-Kameras berechnen

Die drei Kameras liefern echte virtuelle **RGB- und Tiefenbilder**. Das Experiment wandelt Tiefenpixel mittels Kamerakalibrierung in eine **gemeinsame 3D-Punktwolke** um und bestimmt die acht festen AprilTag-Referenzpunkte **aus Bildpixeln**. Erst danach werden die Ergebnisse mit den bekannten MuJoCo-Werten verglichen.

Das ist bewusst **noch keine vollständige 3D-Rekonstruktion aller Wände**: Verdeckte Oberflächen sind nicht beobachtbar, und die virtuellen Kamerapositionen sind bereits ideal kalibriert. Die Abweichungen werden trotzdem numerisch erfasst.

**Neu:** Das MP4-Video blendet die tatsächlichen gemessenen Punkte jeder der drei Kameras schrittweise ein; am Ende werden die rekonstruierten AprilTag-Punkte mit der **nur zur Evaluation verwendeten MuJoCo-Wahrheit** verglichen. Die Videozeit ist eine **animierte Darstellung einer statischen Aufnahme**, keine behauptete Live-SLAM-Messung.


In [ ]:
room=runs/'room'
run_experiment('Drei RGB-D-Kameras rekonstruieren sichtbaren Raum und 8 AprilTags',
    'robobarista.room_reconstruction',
    ['--output',str(room),'--width','960','--height','720','--stride','6'])
display(Image(filename=str(room/'rgbd_room_vs_truth.png')))
display(Image(filename=str(room/'rgbd_room_3d_cloud.png')))
# Video: drei unabhängig aus Bild-Tiefenpixeln berechnete XYZ-Punktwolken
# nacheinander einblenden. Die MuJoCo-Wahrheit erscheint erst am Ende.
run_experiment('3D-Raumaufbau als Video aus den drei RGB-D-Kameras',
    'robobarista.room_reconstruction_video',
    ['--input',str(room),'--output',str(room),
     '--frames','24','--fps','6','--max-points-per-camera','1200'])
show_video(room,'rgbd_room_reconstruction')
rr=json.loads((room/'summary.json').read_text())
print('Feste Marker gemessen:',rr['tag_count_observed'],'/',rr['tag_count_expected'])
print('Gemessene RGB-D-Punkte:',rr['rgbd_points'])
print('Mittlerer Positionsfehler der Marker (mm):',
      round(rr['mean_fixed_tag_position_error_mm'],3))
print('Maximaler Positionsfehler (mm):',
      round(rr['max_fixed_tag_position_error_mm'],3))
print('Tischhoehe: geschätzt (m):',rr['estimated_tabletop_z_m'],
      '  Modell (m):',rr['true_tabletop_z_m'])
display(pd.DataFrame([
    {'Station':r['station'],
     'Cameras':r['camera_observations'],
     'Fehler (mm)':r['error_mm']}
    for r in rr['fixed_station_details']]))


## 2. Der FR5 fährt an die sieben Stationen

Für jede Station werden alternative Gelenkhaltungen für eine **freie Vorposition** (ohne mechanischen Gerätekontakt) gesucht. Dabei prüft die Software den vollständigen Hin- und Rückweg anhand vieler Gelenkzwischenstellungen sowie die konservativ angenommene Form der getragenen Werkzeuge. Nicht sichere Kandidaten werden **verworfen**. Erst zulässige Anfahrten erscheinen im Film.


In [ ]:
staging=runs/'staging'
run_experiment('Sieben kollisionsgepruefte FR5-Anfahrten',
    'robobarista.safe_station_staging_demo',
    ['--seed',str(SEED),'--frames-per-station','12','--fps','6',
     '--width','960','--height','720','--output',str(staging)])
show_video(staging,'safe_staging')
st=json.loads((staging/'safe_staging_report.json').read_text())
print('Sichere VORPOSITIONEN:',st['pre_dock_allowed_count'],'/7')
print('Noch gesperrte Vorpositionen:',st['blocked'])
print('Mechanische DOCKING-Kontakte freigegeben:',
      st['actual_contact_docking_approved'])
display(pd.DataFrame([
    {'Schnittstelle':name,'Ergebnis':v['status'],
     'IK-Posen erreichbar':v['attempts']['ik_reached'],
     'IK-Posen kontaktfrei':v['attempts']['pose_contact_free'],
     'Kollisionsgepruefte Hin-/Rueckwege':
         v['attempts']['safe_full_joint_routes']}
    for name,v in st['stage_poses'].items()]))


## NEU: kompletter Espresso–Milch–Latte-Art-Ablauf (virtueller **Bewegungs-Storyboard**)

Der FR5 fährt in der Reihenfolge **Siebträger → Mühle mit integriertem Tamper → Brühgruppe → Kunststoffbecher → Espressobezug → Milchkanne → Milchspender → Dampflanze → Bechertransport → angenommene Latte-Art-Gießgeste → Reinigung**.

**Was die Bewegung wirklich leistet:** Sieben bekannte Geräte-**Vorpositionen** und ihre Hin-/Rückwege werden für das jeweils virtuelle Werkzeug erneut auf Roboter-, Geräte- und Kannenkollisionen geprüft. Der Film zeigt FR5-Bewegungen. **Das tatsächliche Greifen, Einspannen, Befüllen, Schäumen und fließende Milch ist nicht physikalisch implementiert**: Das sind explizit gekennzeichnete symbolische Aktionen. Auch die angenommene Gießgeste ist ein kleines sicheres Schwenk-/Neigemuster und noch keine resultierende Latte Art.


In [ ]:
coffee=runs/'coffee_process'
run_experiment('FR5-Story: Espresso, integrierter Tamper, Milch, Schaum, Latte-Art',
    'robobarista.coffee_workflow',
    ['--output',str(coffee),'--seed',str(SEED),
     '--width','760','--height','570','--frames-leg','4'])
show_video(coffee,'coffee_workflow')
cw=json.loads((coffee/'summary.json').read_text())
print('Geprüfte Schritte:',cw['safe_predock_stages'],'/',
      cw['virtual_espresso_and_milk_workflow_stages'])
print('Geometrisch angenommene Gießgeste:',cw['assumed_synthetic_wrist_latte_pattern'])
print('Physikalischer Gerätekontakt schon implementiert:',
      cw['true_machine_docking_completed'])
print('Echte Milchschaum-/Espresso-Flüssigkeitsphysik:',
      cw['true_milk_frothing_or_espresso_flow'])
display(pd.DataFrame(json.loads((coffee/'recipe.json').read_text()))[
    ['label','tool','target_dock','kinematic_predock_checked',
     'contact_executed']])


## Zwei geplante FR5-Aufsätze (Modelle aus bestehenden Robotikbibliotheken)

- **Milchkanne/Siebträger:** Robotiq 2F-85, schmale weiche Finger für die Handgriffe. Bereits als Menagerie-MJCF verfügbar.
- **Kunststoffbecher:** Robotiq 2F-140 mit breiten, weichen Kontaktpads, damit die Becherwand nicht punktuell zusammengedrückt wird; als robosuite-Gripper verfügbar.

Diese Zelle dokumentiert **die Tool-Auswahl im Ablauf**. Beide echten Gripper-MJCF-Modelle sind **noch nicht an den FR5 montiert und die Griffkontakte sind nicht physikalisch simuliert**. Flanschadapter, Werkzeugwechsel, Endeffektor-Kollisionshüllen und echte Kräfte sind offene Validierungsaufgaben.


In [ ]:
from robobarista.tool_heads import export_profiles
profiles=export_profiles()
display(pd.DataFrame([
    {'Aufsatz':key,
     'Library-Modell':cfg['library_identifier'],
     'Nennöffnung (mm)':cfg['manufacturer_nominal_opening_mm'],
     'Griffobjekte':', '.join(cfg['grasped_objects'])}
    for key,cfg in profiles['heads'].items()]))
display(pd.DataFrame([
    {'Kaffeeschritt':key,'geplanter Aufsatz':value or 'kein Aufsatz'}
    for key,value in profiles['workflow_tool_selection'].items()]))
print('Echte Greifer im FR5-Modell montiert:',
      not all(not x['model_integrated_in_fr5']
              for x in profiles['heads'].values()))
print('Automatischer Werkzeugwechsel implementiert:',
      profiles['automatic_tool_changer_simulated'])


## 3. Drei Kameras filmen mit **15 FPS** (statt 5 FPS) und sehen eine **neue zufällige erlaubte Bewegung**

Die Bewegung wird vom Programm mit einem festen Seed generiert und vorab auf Arm-/Werkzeugkollisionen geprüft. Eine separate virtuelle Milchkanne führt sie vor. **Nur die RGB-D-Bilder gehen in die Kameraschätzung ein**: AprilTag #71 / #72 werden in Pixeln gesucht und per Mehrkamera-Optimierung in eine 6D-Pose umgerechnet. Der FR5 bekommt nur diese Messungen und folgt ihnen mit inverser Kinematik. Verdeckungen ohne mindestens zwei Kameras stoppen den Replay, anstatt die tatsächlichen Simulator-Koordinaten heimlich zu verwenden.

Wir vergleichen anschließend getrennt **Simulatorwahrheit → Kameramessung → FR5-Gelenkstellung**. Die optische Messung ist nicht identisch mit einer eingeblendeten X-Markierung!


In [ ]:
camera=runs/'camera'
run_experiment('Drei Kameras AprilTag → gemessene 6D-Bahn → FR5-Replay',
    'robobarista.camera_joint_replay',
    ['--seed',str(SEED),'--frames','31','--fps','15',
     '--width','960','--height','720','--output',str(camera)])
print('Links die getrennte optische Vorfuehrung, rechts FR5 nur aus Kameramesswerten.')
show_video(camera,'camera_vs_fr5_side_by_side','demo_and_replay',width=960)
chart=camera/'camera_fr5_trajectory_comparison.png'
if chart.is_file():display(Image(filename=str(chart)))
s=json.loads((camera/'summary.json').read_text())
print('Kamerafrequenz: 15 FPS, optische Bilder: 31 (frueher 5 FPS, 13 Bilder)')
print('Kamera-Pose gueltig:',s['camera_valid_frames'],'/',s['frames'])
print('FR5-Pose aus Kameramessung nachgefahren:',s['replayed_frames'],'/',s['frames'])
print('Max. Kamerapose vs getrennte Simulatorwahrheit (mm):',
      round(s['max_camera_vs_truth_mm'],3))
print('Max. Kamerapose vs getrennte Simulatorwahrheit (°):',
      round(s['max_camera_vs_truth_deg'],3))
print('Feste AprilTags aus Bildern optisch erkannt:',
      s['fixed_station_tags_recognized_in_initial_render'],'/',
      s['fixed_station_reference_count'])
print('Wichtig: Diese Werte stammen aus perfekten virtuellen Kamerakalibrierungen.')


## 3a. Kontinuierliche 6D-Bewegung aus den Kamerabildern

Die 31 **tatsächlich optisch rekonstruierten** AprilTag-Posen mit Zeitstempeln werden zu einer kontinuierlichen Bahn verarbeitet. Drei kubische **Glättungssplines** beschreiben X/Y/Z einschließlich Geschwindigkeit und Beschleunigung, ein **SO(3)-RotationSpline** die Kannenrotation, Winkelgeschwindigkeit und Winkelbeschleunigung.

Die komplette 6D-Bahn lässt sich anschließend mit **100 Hz** abfragen. Das ist eine mathematisch kontinuierliche Bewegung aus **15-FPS-Messungen** – keine 100 echte Kamerabilder pro Sekunde. Die Winkel-Spline interpoliert die gemessenen Rotationen und glättet ihre Messstörungen noch nicht eigenständig.

## 3b. Ruckig plant eine **ruckbegrenzte, durchgehende Gelenkbewegung**

Wir berechnen die Armkonfigurationen an ausgewählten Punkten der kontinuierlichen Kamerakurve, bestimmen dort **gemeinsame (auch von null verschiedene) Gelenkgeschwindigkeiten und Beschleunigungen** und verbinden sie mit **Ruckig**. Die Planung begrenzt Gelenkgeschwindigkeit, Beschleunigung und Jerk (Änderung der Beschleunigung).

**Wichtig:** Wir verwenden ausschließlich die lokalen, freien Ruckig-*State-to-State*-Berechnungen, **keine Cloud-API für Ruckig-Zwischenwegpunkte**. Wenn eine Teilbewegung länger braucht, wird die Dauer transparent verlängert. Jeden geplanten Gelenkzwischenzustand prüfen wir erneut auf Roboterkontakte und auf die komplette angenommene Milchkanne. Die animierte Bahn ist zunächst **kinematisch**, kein Motorleistungsnachweis.



In [ ]:
# NEU: Aus 15 FPS AprilTag → kontinuierliche 6D-Pose → Ruckig mit Jerk-Grenzen
smooth=runs/'continuous'
run_experiment('6D-Splines + lokale Ruckig-Jerk-Bewegungsplanung',
    'robobarista.continuous_motion',
    ['--input',str(camera),'--output',str(smooth),
     '--spline-fps','100','--smooth-mm','.35',
     '--knot-stride','3','--dt','.02'])
show_video(smooth,'continuous_ruckig')
if (smooth/'continuous_6d_ruckig.png').exists():
    display(Image(filename=str(smooth/'continuous_6d_ruckig.png')))
c=json.loads((smooth/'summary.json').read_text())
print('Kamera-FPS:',round(c['input_camera_fps'],2))
print('Kamera-Frames:',c['camera_frames'],
      '→ kontinuierliche Spline-Abfragen:',c['spline_samples'])
print('Lokale Ruckig-Segmente:',c['ruckig_segments'])
print('Hat nicht-null Geschwindigkeiten an Zwischenpunkten:',
      c['nonzero_interior_waypoint_velocity'])
print('Urspruengliche Messdauer:',round(c['source_duration_s'],2),'s')
print('Ruckig-Zielzeit (neu):',round(c['joint_trajectory_duration_s'],2),'s')
print('Gelenk-/Milchkannen-Zwischenpruefungen:',
      c['checked_joint_and_full_pitcher_states'])
print('WARNUNG: Virtuelle Planungsgrenzen, keine zertifizierten FR5-Herstellerlimits.')


## 4. Separater bisheriger Motor-Test: Kann der FR5 der optisch gemessenen Bahn auch **zeitlich** folgen?

Dieser neue Test nutzt die **sechs Drehmoment-Aktuatoren des FR5-Modells** und `mujoco.mj_step()` statt direktem Einsetzen der Gelenkwinkel. Die vorherige optisch gewonnene Gelenkbahn ist der Sollverlauf; für den ersten Regler wird dieser aus Vorsicht mit einer **4-fach langsameren Zeitachse** verfolgt. Ein einfacher PD-Regler mit Schwerkraftkompensation und Drehmomentbegrenzung versucht, ihn zu verfolgen. Der Film zeigt die **tatsächliche dynamische Gelenkbewegung** und der Bericht mögliche Verzögerungen oder Kollisionen.

Das ist ein Test im **virtuellen** MuJoCo-Motormodell und ausdrücklich **noch keine echte Roboterregelung/Sicherheitsfreigabe**. Bei einem Kontaktfehler oder numerischen Abbruch wird der Versuch als fehlgeschlagen markiert.

**Abgrenzung:** Dieser vorhandene Motorvergleich fährt weiterhin die direkt aus den Kameramessungen berechnete Gelenkbahn ab. Die neue **Ruckig-Ausführung oben** ist vorerst eine offline geprüfte Gelenktrajektorie, die noch nicht als Motor-Sollwerte integriert wurde. Wir behaupten also nicht, dass das neue glatte Profil schon physikalisch motorisch getestet ist.


In [ ]:
motor=runs/'motor'
run_experiment('MuJoCo echte Torque-Motorsteuerung mit Zeitschritten',
    'robobarista.motor_follow',
    ['--input',str(camera),'--output',str(motor),
     '--width','960','--height','720','--fps','7',
     '--time-scale',str(TIME_SCALE)])
show_video(motor,'motor_tracking')
ms=json.loads((motor/'summary.json').read_text())
print('Motorstatus:',ms['status'])
print('Motoren mit mj_step:',ms['six_fr5_real_torque_actuators_used'])
print('Max. Folgefehler zur gewuenschten Kinematik (mm):',
    round(ms['max_motor_tracking_position_lag_mm'],2))
print('Mittlerer Folgefehler (mm):',
    round(ms['mean_motor_tracking_position_lag_mm'],2))
print('Motorbewegung zeitlich gestreckt um Faktor:', ms['time_scale'])
print('Dynamische Simulationsdauer (s):',ms['actual_simulated_duration_s'])
print('Maximale Gelenkgeschwindigkeit (rad/s):',round(ms['max_joint_velocity_rad_s'],3))
print('Zeitschritte mit Drehmoment-Begrenzung:',
    ms['torque_saturation_control_intervals'])
print('Sicherheitsverletzung:',ms['first_safety_violation'])


## 5. Dateien und nächste reale Bauaufgaben

Die Videos und alle JSON-Messungen exportieren wir zusammen. Du kannst die **drei verschiedenen Fehlerquellen** weiter auseinanderhalten:

- *Optisch:* Simulatorposition minus AprilTag-Schätzung.
- *Kinematisch:* gemessene Pose minus durch sechs Gelenke erreichter Sollpose.
- *Dynamisch:* gewünschte Gelenkposition minus tatsächlich motorisch gefahrener Gelenkposition zum gleichen virtuellen Zeitpunkt.

Die komplette Arbeitszelle steht, aber **mechanische Andockkontakte in Geräten, eine wirklich gegriffene Kanne, echtes Temperieren der Milch, ein Siebträger-Bajonett und Flüssigkeitsphysik bleiben offene Entwicklungsschritte.** Die virtuellen AprilTag-Kameraparameter sind noch nicht aus realen Fotos kalibriert; die 30-mm-Kollisionshüllen sind Planungsfilter mit Näherungsgeometrien, keine sicherheitsgerichtete Ausrüstung.

Die Zukunft beginnt also von dieser **funktionierenden, zusammengeführten Forschungsbasis**, nicht mit einer Behauptung, das Gerät könne schon einen echten Kaffee kochen.

[Projektcode](https://github.com/Coffee-Tobi/robocasa) · [Technische Erklärung](https://github.com/Coffee-Tobi/robocasa/blob/main/planning/SAFE_CAMERA_REPLAY_EXPLAINED.md)

**Neu 3a/3b:** Die Datei `camera_continuous_6d.json` enthält die 100-Hz-Pose **inklusive Geschwindigkeiten und Beschleunigungen**; `ruckig_joint_trajectory.json` enthält Ruckig-Gelenkpositionen, -geschwindigkeiten und -beschleunigungen mit eigener Zeitachse und `summary.json` die tatsächlich verlängerte Bewegungsdauer. Unsere frühere Motor-Regelung bleibt unverändert und getrennt.


In [ ]:
# 5. Gemeinsames Ergebnisarchiv: alle vier Videos + Messungen
archive=runs/'RoboBarista_Gesamtpaket.zip'
wanted={
    room:['rgbd_room_reconstruction.mp4',
          'rgbd_reconstruction_video_summary.json',
          'rgbd_room_vs_truth.png','rgbd_room_3d_cloud.png',
          'reconstructed_xyz_cloud.npz',
          'summary.json'],
    coffee:['coffee_workflow.mp4','coffee_workflow.gif',
            'recipe.json','summary.json'],
    workcell:['workcell_demo.mp4','layout_topdown.png','summary.json',
              'dock_targets.json','workspace_constraints.json',
              'visibility.json','reachability.json'],
    staging:['safe_staging.mp4','safe_staging_report.json'],
    camera:['camera_vs_fr5_side_by_side.mp4',
            'demo_and_replay.mp4',
            'camera_fr5_trajectory_comparison.png',
            'camera_measurements.json','fixed_reference_vision_check.json',
            'robot_replay.json','ground_truth_evaluation_only.json',
            'camera_vs_truth_EVAL_ONLY.json','summary.json'],
    smooth:['continuous_ruckig.mp4','continuous_6d_ruckig.png',
            'camera_continuous_6d.json','ruckig_joint_trajectory.json','summary.json'],
    motor:['motor_tracking.mp4','motor_tracking.json','summary.json'],
}
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as out_zip:
    for folder,files in wanted.items():
        for name in files:
            file=folder/name
            if file.exists():
                out_zip.write(file,file.relative_to(runs))
print('Archiv:',archive, '(',round(archive.stat().st_size/1e6,1),'MB )')
display(FileLink(str(archive)))
